In [ ]:
import os
import cv2
import numpy as np
from ultralytics import YOLO

script_dir = os.path.dirname(os.path.abspath(os.getcwd()))
VIDEO_PATH = os.path.join(script_dir, "Folder", "Video Path")
MODEL_PATH = "yolov8n.pt"

CONFIDENCE = 0.35

GRID_ROWS = 4
GRID_COLS = 6

model = YOLO(MODEL_PATH)

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise RuntimeError("Failed to Open your Video")

while True:

    success, frame = cap.read()

    if not success:
        break

    height, width = frame.shape[:2]

    results = model(
        frame,
        conf=CONFIDENCE,
        verbose=False
    )

    occupied_cells = set()

    for result in results:

        boxes = result.boxes

        for box in boxes:

            x1, y1, x2, y2 = map(
                int,
                box.xyxy[0].tolist()
            )

            class_id = int(box.cls[0])

            confidence = float(box.conf[0])

            class_name = model.names[class_id]

            for row in range(GRID_ROWS):

                for col in range(GRID_COLS):

                    cell_x1 = col * width // GRID_COLS
                    cell_y1 = row * height // GRID_ROWS

                    cell_x2 = (col + 1) * width // GRID_COLS
                    cell_y2 = (row + 1) * height // GRID_ROWS

                    overlap_x = max(
                        0,
                        min(x2, cell_x2) - max(x1, cell_x1)
                    )

                    overlap_y = max(
                        0,
                        min(y2, cell_y2) - max(y1, cell_y1)
                    )

                    if overlap_x > 0 and overlap_y > 0:
                        occupied_cells.add((row, col))

            cv2.rectangle(
                frame,
                (x1, y1),
                (x2, y2),
                (0, 255, 0),
                2
            )

            label = f"{class_name} {confidence:.2f}"

            cv2.putText(
                frame,
                label,
                (x1, max(y1 - 10, 20)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 255, 0),
                2
            )

    for row in range(GRID_ROWS):

        for col in range(GRID_COLS):

            if (row, col) not in occupied_cells:

                x1 = col * width // GRID_COLS
                y1 = row * height // GRID_ROWS

                x2 = (col + 1) * width // GRID_COLS
                y2 = (row + 1) * height // GRID_ROWS

                cv2.rectangle(
                    frame,
                    (x1, y1),
                    (x2, y2),
                    (255, 180, 0),
                    2
                )

                cv2.putText(
                    frame,
                    "Empty Spot",
                    (x1 + 5, y1 + 25),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    (255, 180, 0),
                    2
                )

    cv2.imshow("Universal Video Detector", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()